# Can a 100-Year-Old Statistic Catch ChatGPT?
## Lexical diversity vs. LLM-generated fake hotel reviews — a reproducible benchmark

**Thesis.** Lexical-diversity statistics (Yule's K, Herdan's C, MTLD, MATTR) are a cheap,
interpretable, near-zero-cost first-line filter that reliably catches *bulk / lazily-generated*
LLM review spam — but degrade against carefully-prompted, human-mimicking output. Knowing
*where* that boundary sits is more useful than any single AUC number.

**Data.** MAiDE-up (`all_data.csv`): ~20k hotel reviews, `source=0` real / `source=1` GPT-generated,
balanced across 10 languages and sentiment.

**Honest framing.** This is *not* a victory lap for old stats. A 2025 result
([arXiv 2506.13313](https://arxiv.org/pdf/2506.13313)) finds well-crafted LLM fake reviews are
*indistinguishable* to humans and machines. We test exactly where the cheap signal holds and where it breaks.

> Everything below is seeded and runnable end-to-end. Metric implementations live in
> `lexical_metrics.py` (unit-tested in `test_lexical_metrics.py`).

## 0. Setup & global seed
One seed, set once, for every RNG we touch.

In [ ]:
import os, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

import pandas as pd
import matplotlib.pyplot as plt

import lexical_metrics as lm

pd.set_option("display.max_colwidth", 80)
DATA_PATH = "../all_data.csv"   # adjust if you run from the repo root
LANG = "English"                # metric-valid subset (space-delimited); see README
print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. Load MAiDE-up and build the label

Label encoding (verified on the file):

| column | meaning |
|---|---|
| `source = 0` | **real** human review (label 0) |
| `source = 1` | **LLM-generated** fake (label 1) |

The review text is split across `Upside_Review` (pros) and `Downside_Review` (cons); we join them
into one `text` field. We restrict to `English` because the lexical metrics assume space-delimited
tokenization — see the README caveat before extending to Chinese/Korean.

In [ ]:
raw = pd.read_csv(DATA_PATH)

def join_text(row):
    parts = [str(row.get("Upside_Review", "") or ""), str(row.get("Downside_Review", "") or "")]
    return " ".join(p.strip() for p in parts if p and p.lower() != "nan").strip()

raw["text"] = raw.apply(join_text, axis=1)
raw["label"] = raw["source"].astype(int)          # 0 = real, 1 = LLM

df = raw[raw["Review_Language"] == LANG].copy()
df = df[df["text"].str.len() > 0].reset_index(drop=True)

print("English rows:", len(df))
print(df["label"].value_counts().rename({0: "real", 1: "llm"}))
df[["Hotel Name", "Sentiment", "label", "text"]].head(3)

## 2. Sanity check the class balance & review length
Guard against the trap where the two classes just differ in length.

In [ ]:
df["n_tokens"] = df["text"].map(lambda t: len(lm.tokenize(t)))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.groupby("label")["n_tokens"].median().rename({0: "real", 1: "llm"}).plot(
    kind="bar", ax=ax[0], color=["#4C72B0", "#DD8452"], rot=0, title="Median tokens per review")
for lbl, name, c in [(0, "real", "#4C72B0"), (1, "llm", "#DD8452")]:
    ax[1].hist(df[df.label == lbl]["n_tokens"], bins=40, alpha=0.6, label=name, color=c)
ax[1].set(title="Review-length distribution", xlabel="tokens"); ax[1].legend()
plt.tight_layout(); plt.show()

print(df.groupby("label")["n_tokens"].describe()[["mean", "50%", "max"]])

## 3. The intuition: richness as a fraud signal (Zipf curves)

Human writing tends to be *bursty* and lexically uneven; bulk-generated text tends toward flatter,
more uniform vocabulary. Overlay the rank-frequency (Zipf) curves for each class to *see* the difference
before we quantify it.

In [ ]:
from collections import Counter

def zipf(tokens_iter):
    c = Counter()
    for toks in tokens_iter:
        c.update(toks)
    freqs = sorted(c.values(), reverse=True)
    return np.arange(1, len(freqs) + 1), np.array(freqs)

plt.figure(figsize=(6, 4.2))
for lbl, name, c in [(0, "real", "#4C72B0"), (1, "llm", "#DD8452")]:
    ranks, freqs = zipf(df[df.label == lbl]["text"].map(lm.tokenize))
    plt.loglog(ranks, freqs, label=name, color=c, lw=1.6)
plt.xlabel("word rank (log)"); plt.ylabel("frequency (log)")
plt.title("Zipf curves: real vs. LLM reviews"); plt.legend(); plt.tight_layout(); plt.show()

## 4. Lexical-diversity features

Metrics are implemented and unit-tested in `lexical_metrics.py`. We compute the full feature vector
per review. `n_tokens` is carried along so we can *ablate* length later and prove the signal is not
just document length in disguise.

> A quick reminder of Yule's K:  $K = 10^4 \cdot (\sum_i f_i^2 - N) / N^2$ — higher means more
> repetition / lower diversity.

In [ ]:
feat_rows = df["text"].map(lm.features).tolist()
feats = pd.DataFrame(feat_rows)
feats["label"] = df["label"].values

FEATURE_COLS = ["ttr", "herdan_c", "yule_k", "mattr", "mtld", "n_tokens"]
feats.groupby("label")[FEATURE_COLS].mean().rename(index={0: "real", 1: "llm"})

In [ ]:
# Distribution of each metric by class -- which signals actually separate?
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), FEATURE_COLS):
    for lbl, name, c in [(0, "real", "#4C72B0"), (1, "llm", "#DD8452")]:
        ax.hist(feats[feats.label == lbl][col], bins=40, alpha=0.55, label=name, color=c, density=True)
    ax.set_title(col); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Experiment 1 — the honest lexical baseline

Lexical features only → simple, interpretable classifiers (logistic regression + gradient boosting),
evaluated on a held-out test set with a fixed seed. Reviews are imbalanced in the wild, so we report
**PR-AUC** and **precision@k** (the trust-and-safety-relevant metric) alongside ROC-AUC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

# Exclude n_tokens from the *diversity* feature set; keep it only for the ablation below.
DIVERSITY_COLS = ["ttr", "herdan_c", "yule_k", "mattr", "mtld"]
X = feats[DIVERSITY_COLS].values
y = feats["label"].values

# Split on INDICES (not arrays) so Experiment 3 can hold out the exact rows the
# model trained on -- otherwise the tier AUCs are inflated by train/eval overlap.
idx = np.arange(len(df))
tr_idx, te_idx = train_test_split(idx, test_size=0.25, random_state=SEED, stratify=y)
X_tr, X_te, y_tr, y_te = X[tr_idx], X[te_idx], y[tr_idx], y[te_idx]
train_texts = set(df.iloc[tr_idx]["text"])   # anything here is excluded from Experiment 3

models = {
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
    "gboost": GradientBoostingClassifier(random_state=SEED),
}

def precision_at_k(y_true, scores, k_frac=0.10):
    k = max(1, int(len(scores) * k_frac))
    idx = np.argsort(scores)[::-1][:k]
    return y_true[idx].mean()

results = {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]
    results[name] = {
        "roc_auc": roc_auc_score(y_te, p),
        "pr_auc": average_precision_score(y_te, p),
        "prec@10%": precision_at_k(y_te, p, 0.10),
        "_scores": p,
    }
pd.DataFrame({k: {m: v[m] for m in ["roc_auc", "pr_auc", "prec@10%"]} for k, v in results.items()}).T

In [ ]:
# ROC + PR curves for the stronger model
best = max(results, key=lambda k: results[k]["pr_auc"])
p = results[best]["_scores"]
fpr, tpr, _ = roc_curve(y_te, p)
prec, rec, _ = precision_recall_curve(y_te, p)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(fpr, tpr, color="#DD8452"); ax[0].plot([0, 1], [0, 1], "--", color="gray", lw=0.8)
ax[0].set(title=f"ROC — {best} (AUC={results[best]['roc_auc']:.3f})", xlabel="FPR", ylabel="TPR")
ax[1].plot(rec, prec, color="#4C72B0")
ax[1].set(title=f"PR — {best} (AP={results[best]['pr_auc']:.3f})", xlabel="Recall", ylabel="Precision")
plt.tight_layout(); plt.show()

# Feature importance / coefficients -> the interpretability payoff
if best == "gboost":
    imp = pd.Series(models["gboost"].feature_importances_, index=DIVERSITY_COLS)
else:
    imp = pd.Series(models["logreg"].named_steps["logisticregression"].coef_[0], index=DIVERSITY_COLS)
imp.sort_values().plot(kind="barh", figsize=(6, 3), color="#55A868", title=f"{best} feature weights"); plt.tight_layout(); plt.show()

## 6. Length ablation — is it diversity, or just length?

The credibility check. We retrain (a) on `n_tokens` alone and (b) on diversity features with length
regressed out / included, and compare. If a length-only model matches the diversity model, our
"lexical diversity works" story collapses into "LLM reviews are just a different length." Report both.

In [ ]:
# (a) length-only baseline
Xl = feats[["n_tokens"]].values
Xl_tr, Xl_te, _, _ = train_test_split(Xl, y, test_size=0.25, random_state=SEED, stratify=y)
len_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)).fit(Xl_tr, y_tr)
p_len = len_model.predict_proba(Xl_te)[:, 1]

# (b) diversity + length together
Xb = feats[DIVERSITY_COLS + ["n_tokens"]].values
Xb_tr, Xb_te, _, _ = train_test_split(Xb, y, test_size=0.25, random_state=SEED, stratify=y)
both = GradientBoostingClassifier(random_state=SEED).fit(Xb_tr, y_tr)
p_both = both.predict_proba(Xb_te)[:, 1]

pd.DataFrame({
    "length_only":       [roc_auc_score(y_te, p_len),  average_precision_score(y_te, p_len)],
    "diversity_only":    [results[best]["roc_auc"],    results[best]["pr_auc"]],
    "diversity+length":  [roc_auc_score(y_te, p_both), average_precision_score(y_te, p_both)],
}, index=["roc_auc", "pr_auc"]).T

## 7. Experiment 2 — the neural detector it's up against  *(stub)*

Fine-tune / apply a small transformer (DistilBERT or RoBERTa, or an off-the-shelf AI-text detector)
on the **same split, same seed, same test set** — that fairness is the credibility of the whole piece.
Report identical metrics **plus cost/latency** (ms/review, params): this is where the cheap method
"wins" even when AUC loses.

```python
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# ... tokenize df['text'] with the SAME X_tr/X_te indices, fine-tune, evaluate ...
# record: roc_auc, pr_auc, prec@10%, inference_ms_per_review, n_params
```
Leave this as the next work item — Experiment 1 already stands on its own.

## 8. Experiment 3 — who actually evades the detector

MAiDE-up's fakes are one style from one model (GPT). To probe generalization we generate fresh fakes
over held-out hotels across a **capability gradient AND a vendor boundary**, and score them with the
**same trained lexical model** from Experiment 1:

| tier | model | prompt |
|---|---|---|
| **lazy** | Haiku 4.5 (Claude) | terse one-line prompt (what a spammer types) |
| **careful** | Sonnet 5 (Claude) | persona prompt ("write like a real guest") |
| **frontier** | Opus 4.8 (Claude) | rich red-team persona + adaptive thinking |
| **crossvendor** | Llama 3.3 70B (Meta, via Groq) | `careful` prompt, different vendor |

Generate the tiers first (outside the notebook, a few \$ of API — needs `ANTHROPIC_API_KEY` and, for
the cross-vendor tier, `GROQ_API_KEY`):

```bash
python ../generate_difficulty_tiers.py --n 40   # writes difficulty_tiers.csv
```

**The intuitive expectation — better model, harder to catch — is wrong, and the cross-vendor tier
sharpens it.** The detector flags *sustained richness* as fake, so the more a model writes, the more it
gives itself away: within Claude, Opus > Sonnet > lazy in AUC; and Meta's **Llama, the wordiest of all,
is caught *most* easily (~0.88, above in-distribution GPT)**. The tell generalizes across vendors. The
one universal evader is the **terse lazy tier**, whose brevity drops its diversity *below* real humans.
This is a **verbosity detector** — the ranking transfers across vendors even though the scores don't.

> ### 🔌 No API key? Reproduce the pipeline offline
>
> The `--offline` flag builds the tiers from the **local dataset** so the scoring cells still run with
> no credentials:
>
> ```bash
> python ../generate_difficulty_tiers.py --offline --n 40
> ```
>
> Note this is a **plumbing stand-in, not the real result** — offline, `lazy` is hand-written
> templates and `frontier` is real human text relabeled as fake, which produces a *different* (peaked)
> AUC shape. The real API run above is what the numbers below and in the write-up reflect.

In [ ]:
import os

TIERS_PATH = "difficulty_tiers.csv"
assert os.path.exists(TIERS_PATH), (
    "Create difficulty_tiers.csv first:\n"
    "  real:    python ../generate_difficulty_tiers.py --n 40           (needs ANTHROPIC_API_KEY)\n"
    "  offline: python ../generate_difficulty_tiers.py --offline --n 40 (no API, demo stand-in)"
)

tiers = pd.read_csv(TIERS_PATH)
tiers["text"] = tiers["text"].astype(str)
# Drop empties AND any fake whose text was in the training set (honest eval:
# no evaluated example may also be a training example).
tiers = tiers[(tiers["text"].str.len() > 0) & (~tiers["text"].isin(train_texts))].reset_index(drop=True)

# Lexical features for every generated fake, using the SAME metric code.
tier_feats = pd.DataFrame(tiers["text"].map(lm.features).tolist())
tier_feats["tier"] = tiers["tier"].values
print(tiers["tier"].value_counts())
tier_feats.groupby("tier")[DIVERSITY_COLS].mean()

In [ ]:
# Score each tier with the trained Experiment-1 model. For a tier's AUC we pit its
# fakes (label 1) against the SAME pool of real reviews (label 0) every time, so the
# only thing that changes across tiers is how hard the fakes are to catch.
trained = models[best]                      # fitted on DIVERSITY_COLS in Experiment 1
# Negatives = HELD-OUT real reviews only (never seen in training).
held_out_real = feats.iloc[te_idx]
real = held_out_real[held_out_real.label == 0][DIVERSITY_COLS].values
real_y = np.zeros(len(real))

TIER_ORDER = ["lazy", "careful", "frontier", "crossvendor"]
auc_by_tier = {}
for t in TIER_ORDER:
    fake = tier_feats[tier_feats.tier == t][DIVERSITY_COLS].values
    if len(fake) == 0:
        continue
    X_eval = np.vstack([real, fake])
    y_eval = np.concatenate([real_y, np.ones(len(fake))])
    scores = trained.predict_proba(X_eval)[:, 1]
    auc_by_tier[t] = roc_auc_score(y_eval, scores)

pd.Series(auc_by_tier).rename("roc_auc").to_frame()

In [ ]:
# The Claude capability gradient as a line; the cross-vendor Llama tier as a
# distinct point at the 'careful' x-position (matched prompt effort). On the real
# run Llama is caught MOST easily (above in-distribution), while the terse lazy
# tier evades most -- brevity, not vendor, is the blind spot.
claude = [t for t in ["lazy", "careful", "frontier"] if t in auc_by_tier]
cvals = [auc_by_tier[t] for t in claude]

plt.figure(figsize=(6.2, 4.2))
plt.plot(claude, cvals, "o-", color="#DD8452", lw=2, ms=9, label="Claude tiers")
for x, v in zip(claude, cvals):
    plt.annotate(f"{v:.2f}", (x, v), textcoords="offset points", xytext=(0, 10), ha="center")
if "crossvendor" in auc_by_tier:
    plt.scatter(["careful"], [auc_by_tier["crossvendor"]], color="#55A868", marker="D", s=90, zorder=5,
                label="Llama (cross-vendor, careful prompt)")
    plt.annotate(f"{auc_by_tier['crossvendor']:.2f}", ("careful", auc_by_tier["crossvendor"]),
                 textcoords="offset points", xytext=(8, -4), ha="left", color="#3d7a4e")
plt.axhline(0.5, ls="--", color="gray", lw=0.9, label="chance (AUC=0.5)")
plt.axhline(results[best]["roc"], ls=":", color="#4C72B0", lw=1.1, label="in-distribution (GPT)")
plt.ylim(0.5, 0.95)
plt.ylabel("lexical detector ROC-AUC")
plt.xlabel("generator:  lazy=Haiku   careful=Sonnet   frontier=Opus")
plt.title("Cross-vendor: verbose fakes are caught, terse ones evade")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## 9. Takeaway (write this last, honestly)

The claim to defend is **not** "old stats beat neural nets." It is:

> Interpretable, near-free statistics still do real work as a **first-stage filter** against
> machine-fluent LLM spam — 0.81 in-distribution, 96% precision on the top decile, and the *ranking*
> holds across three vendors: wordy machine text is caught, whoever wrote it. But what you've built is
> a **verbosity detector**, not an AI-text detector. Its blind spot isn't the frontier model — a fresh
> vendor's fake (Llama) was the *easiest* thing to catch — it's **brevity**. What slips past is a
> spammer who tells any model to keep it short.

Ship the cascade design (cheap pre-filter → escalate the terse, ambiguous middle to the expensive
model), and state the limitations prominently (scores don't transfer across generators — only the
ranking does; the 2025 "indistinguishable" ceiling; English-only metric validity; Goodhart — which
here is as easy as writing less).